# Парсинг статей с Хабра (+ PDF)

Ноутбук выполняет:
- загрузку статьи Хабра по URL;
- извлечение заголовка, автора, даты, тегов и основного текста;
- сохранение результата в TXT и JSON;
- создание PDF-файла со статьёй;
- пакетную обработку нескольких URL.

**Важно:** сайт Хабра может менять HTML-разметку или ограничивать автоматические запросы. При необходимости используйте задержку между запросами.

In [ ]:
!pip -q install requests beautifulsoup4 lxml reportlab


In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urlparse
import json, re, time, html

OUTPUT_DIR = Path('habr_results')
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/150.0 Safari/537.36'
}


In [ ]:
def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def slugify(text):
    text = re.sub(r'[^\w\s-]', '', text, flags=re.UNICODE)
    return re.sub(r'[-\s]+', '_', text).strip('_')[:100]

def parse_habr(url, timeout=30):
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')

    title = ''
    h1 = soup.select_one('h1.tm-title, h1')
    if h1:
        title = clean_text(h1.get_text(' ', strip=True))

    author = ''
    author_el = soup.select_one('.tm-user-info__username, a.tm-user-info__username, a.tm-user-info__userpic')
    if author_el:
        author = clean_text(author_el.get_text(' ', strip=True))

    date = ''
    time_el = soup.select_one('time')
    if time_el:
        date = time_el.get('datetime') or clean_text(time_el.get_text(' ', strip=True))

    article = soup.select_one('div.article-formatted-body')
    if article is None:
        article = soup.select_one('div.tm-article-body')
    if article is None:
        article = soup.select_one('article')

    paragraphs = []
    if article:
        for element in article.select('script, style, noscript'):
            element.decompose()
        for p in article.select('p, h2, h3, li, blockquote'):
            value = clean_text(p.get_text(' ', strip=True))
            if value:
                paragraphs.append(value)

    text = '\n\n'.join(paragraphs)

    tags = []
    for tag in soup.select('a.tm-tags-list__link, a.tm-tags-list__link-container'):
        value = clean_text(tag.get_text(' ', strip=True))
        if value and value not in tags:
            tags.append(value)

    return {
        'url': url,
        'title': title,
        'author': author,
        'date': date,
        'tags': tags,
        'text': text,
        'characters': len(text),
        'paragraphs': len(paragraphs)
    }


In [ ]:
def save_json(data, path):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')

def save_txt(data, path):
    content = [
        data.get('title', ''),
        f"Автор: {data.get('author', '')}",
        f"Дата: {data.get('date', '')}",
        f"URL: {data.get('url', '')}",
        f"Теги: {', '.join(data.get('tags', []))}",
        '',
        data.get('text', '')
    ]
    Path(path).write_text('\n'.join(content), encoding='utf-8')


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_LEFT

def make_pdf(data, path):
    # Пытаемся использовать Unicode-шрифт Windows для русского текста.
    font_candidates = [
        ('Arial', r'C:\\Windows\\Fonts\\arial.ttf'),
        ('DejaVuSans', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    ]
    font_name = 'Helvetica'
    for name, font_path in font_candidates:
        if os.path.exists(font_path):
            pdfmetrics.registerFont(TTFont(name, font_path))
            font_name = name
            break

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('TitleRU', parent=styles['Title'], fontName=font_name, fontSize=16, leading=20)
    body_style = ParagraphStyle('BodyRU', parent=styles['BodyText'], fontName=font_name, fontSize=10, leading=14)
    meta_style = ParagraphStyle('MetaRU', parent=styles['BodyText'], fontName=font_name, fontSize=9, leading=12)

    doc = SimpleDocTemplate(str(path), pagesize=A4, rightMargin=45, leftMargin=45, topMargin=45, bottomMargin=45)
    story = [Paragraph(html.escape(data.get('title', 'Без названия')), title_style), Spacer(1, 12)]
    meta = f"Автор: {html.escape(data.get('author',''))}<br/>Дата: {html.escape(data.get('date',''))}<br/>URL: {html.escape(data.get('url',''))}<br/>Теги: {html.escape(', '.join(data.get('tags', [])))}"
    story += [Paragraph(meta, meta_style), Spacer(1, 14)]
    for paragraph in data.get('text','').split('\n\n'):
        if paragraph.strip():
            story += [Paragraph(html.escape(paragraph), body_style), Spacer(1, 7)]
    doc.build(story)


In [ ]:
# Пример парсинга одной статьи Хабра
url = 'https://habr.com/ru/companies/ruvds/articles/994166/'

article = parse_habr(url)
name = slugify(article['title']) or 'habr_article'

json_path = OUTPUT_DIR / f'{name}.json'
txt_path = OUTPUT_DIR / f'{name}.txt'
pdf_path = OUTPUT_DIR / f'{name}.pdf'

save_json(article, json_path)
save_txt(article, txt_path)
make_pdf(article, pdf_path)

print('Заголовок:', article['title'])
print('Автор:', article['author'])
print('Дата:', article['date'])
print('Абзацев:', article['paragraphs'])
print('Символов:', article['characters'])
print('Файлы:', json_path, txt_path, pdf_path)


In [ ]:
# Пакетный парсинг нескольких статей
urls = [
    'https://habr.com/ru/companies/ruvds/articles/994166/',
    # Добавьте сюда другие ссылки Хабра:
    # 'https://habr.com/ru/articles/XXXXXXXX/',
]

results = []
for url in urls:
    try:
        data = parse_habr(url)
        name = slugify(data['title']) or 'habr_article'
        save_json(data, OUTPUT_DIR / f'{name}.json')
        save_txt(data, OUTPUT_DIR / f'{name}.txt')
        make_pdf(data, OUTPUT_DIR / f'{name}.pdf')
        results.append(data)
        print(f'OK: {data["title"]}')
    except Exception as e:
        print(f'Ошибка при обработке {url}: {e}')
    time.sleep(2)

print(f'Обработано статей: {len(results)}')


## Что получается в результате

В папке `habr_results` для каждой статьи создаются:

- `.json` — структурированные данные;
- `.txt` — полный текст;
- `.pdf` — версия статьи в PDF.

Для новой статьи достаточно добавить её URL в список `urls` и запустить ячейку пакетного парсинга.